# Wood Cutting Problem in Pure Python
## Exact Techniques for Book Problem 3.2

This notebook solves the frame-cutting problem from book section `3.2` in pure Python.

The textbook keeps only the efficient cutting patterns:
- pattern `1`: `(119, 119)` with `62 cm` of waste,
- pattern `2`: `(119, 90, 90)` with `1 cm` of waste,
- pattern `3`: `(90, 90, 90)` with `30 cm` of waste.

We compare three exact approaches:
1. direct enumeration of the pattern counts,
2. reduced search using the exact `119 cm` balance,
3. dynamic programming on the remaining demand.


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from time import perf_counter


In [2]:
def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


PATTERNS = {
    1: {"pieces_119": 2, "pieces_90": 0, "waste": 62},
    2: {"pieces_119": 1, "pieces_90": 2, "waste": 1},
    3: {"pieces_119": 0, "pieces_90": 3, "waste": 30},
}
DEMAND = {"pieces_119": 350, "pieces_90": 350}
INVENTORY_COST = {"pieces_119": 40, "pieces_90": 30}
WASTE_COST = 1
EXPECTED = {
    "pattern_1": 87,
    "pattern_2": 176,
    "pattern_3": 0,
    "inventory_119": 0,
    "inventory_90": 2,
    "cost": 5_630,
}


def make_solution(pattern_1, pattern_2, pattern_3):
    produced_119 = 2 * pattern_1 + pattern_2
    produced_90 = 2 * pattern_2 + 3 * pattern_3
    if produced_119 < DEMAND["pieces_119"] or produced_90 < DEMAND["pieces_90"]:
        return None
    inventory_119 = produced_119 - DEMAND["pieces_119"]
    inventory_90 = produced_90 - DEMAND["pieces_90"]
    cost = (
        pattern_1 * PATTERNS[1]["waste"] * WASTE_COST
        + pattern_2 * PATTERNS[2]["waste"] * WASTE_COST
        + pattern_3 * PATTERNS[3]["waste"] * WASTE_COST
        + inventory_119 * INVENTORY_COST["pieces_119"]
        + inventory_90 * INVENTORY_COST["pieces_90"]
    )
    return {
        "pattern_1": pattern_1,
        "pattern_2": pattern_2,
        "pattern_3": pattern_3,
        "inventory_119": inventory_119,
        "inventory_90": inventory_90,
        "cost": cost,
    }


def choose_better(left, right):
    if left is None:
        return right
    if right is None:
        return left
    left_key = (left["cost"], left["pattern_1"], left["pattern_2"], left["pattern_3"])
    right_key = (right["cost"], right["pattern_1"], right["pattern_2"], right["pattern_3"])
    return right if right_key < left_key else left


## 1. Direct Enumeration


In [3]:
@timed
def solve_cutting_naive():
    best = None
    for pattern_1 in range(0, DEMAND["pieces_119"] // 2 + 1):
        for pattern_2 in range(0, DEMAND["pieces_119"] + 1):
            produced_119 = 2 * pattern_1 + pattern_2
            if produced_119 < DEMAND["pieces_119"]:
                continue
            remaining_90 = max(0, DEMAND["pieces_90"] - 2 * pattern_2)
            max_pattern_3 = remaining_90 // 3 + 2
            for pattern_3 in range(0, max_pattern_3 + 1):
                best = choose_better(best, make_solution(pattern_1, pattern_2, pattern_3))
    return best


naive_result, naive_time = solve_cutting_naive()
print("Naive result:", naive_result)
print(f"Elapsed time: {naive_time:.6f} seconds")
assert naive_result == EXPECTED


Naive result: {'pattern_1': 87, 'pattern_2': 176, 'pattern_3': 0, 'inventory_119': 0, 'inventory_90': 2, 'cost': 5630}
Elapsed time: 0.112996 seconds


## 2. Reduced Search


In [4]:
@timed
def solve_cutting_reduced_search():
    best = None
    for pattern_1 in range(0, DEMAND["pieces_119"] // 2 + 1):
        pattern_2 = DEMAND["pieces_119"] - 2 * pattern_1
        if pattern_2 < 0:
            continue
        remaining_90 = max(0, DEMAND["pieces_90"] - 2 * pattern_2)
        pattern_3 = (remaining_90 + 2) // 3
        best = choose_better(best, make_solution(pattern_1, pattern_2, pattern_3))
    return best


reduced_result, reduced_time = solve_cutting_reduced_search()
print("Reduced-search result:", reduced_result)
print(f"Elapsed time: {reduced_time:.6f} seconds")
assert reduced_result == EXPECTED


Reduced-search result: {'pattern_1': 87, 'pattern_2': 176, 'pattern_3': 0, 'inventory_119': 0, 'inventory_90': 2, 'cost': 5630}
Elapsed time: 0.000137 seconds


## 3. Dynamic Programming


In [5]:
@timed
def solve_cutting_dynamic_programming():
    @lru_cache(maxsize=None)
    def dp(pattern_1):
        if pattern_1 > DEMAND["pieces_119"] // 2:
            return None
        pattern_2 = DEMAND["pieces_119"] - 2 * pattern_1
        if pattern_2 < 0:
            return None
        remaining_90 = max(0, DEMAND["pieces_90"] - 2 * pattern_2)
        pattern_3 = (remaining_90 + 2) // 3
        current = make_solution(pattern_1, pattern_2, pattern_3)
        tail = dp(pattern_1 + 1)
        return choose_better(current, tail)

    return dp(0)


dp_result, dp_time = solve_cutting_dynamic_programming()
print("Dynamic-programming result:", dp_result)
print(f"Elapsed time: {dp_time:.6f} seconds")
assert dp_result == EXPECTED

print("\nAll Python techniques agree with the textbook cutting plan.")


Dynamic-programming result: {'pattern_1': 87, 'pattern_2': 176, 'pattern_3': 0, 'inventory_119': 0, 'inventory_90': 2, 'cost': 5630}
Elapsed time: 0.000237 seconds

All Python techniques agree with the textbook cutting plan.


## Final Check

The optimal plan cuts `87` rods with pattern `1`, `176` rods with pattern `2`, and no rods with pattern `3`. The only excess is `2` pieces of `90 cm`, giving the published total cost of `5,630` USD.
